In [99]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [100]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [101]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [102]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [103]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [104]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [105]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [106]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [107]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [108]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [109]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [110]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [111]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [112]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [113]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [114]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [115]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [116]:
# Choose features from the result of Cox PLSR in R
plsr = [
"shape_Elongation",
"shape_MajorAxisLength",
"shape_Sphericity",
"shape_SurfaceVolumeRatio",
"shape_Flatness",
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16",
"shape_MinorAxisLength",
"glszm_ZonePercentage_PET_b2",
"LBP_201_PET",
"shape_Maximum3DDiameter",
"glszm_GrayLevelVariance_PET_b2"
]


In [117]:
X_plsr = X.loc[:, plsr]
X_new = X_plsr.copy()

In [118]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [119]:
# Copy the original X for later 
original_X = X.copy()

In [133]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [134]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [135]:
X_new

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_MinorAxisLength,glszm_ZonePercentage_PET_b2,LBP_201_PET,shape_Maximum3DDiameter,glszm_GrayLevelVariance_PET_b2
0,0.600926,42.073251,0.761164,0.251218,0.535140,0.029425,25.282894,0.001048,0.000062,47.339202,5.633218
1,0.841579,24.613845,0.697049,0.489853,0.367109,0.037915,20.714498,0.002794,0.000349,28.106939,1.187500
2,0.772821,48.030294,0.565792,0.278467,0.597785,0.008009,37.118833,0.000690,0.000000,60.049979,1.310000
3,0.847727,25.589900,0.684364,0.474018,0.405730,0.018398,21.693241,0.000899,0.000000,32.572995,0.666667
4,0.831483,34.684750,0.503142,0.563135,0.442406,0.013051,28.839789,0.002591,0.000399,39.962482,1.076923
...,...,...,...,...,...,...,...,...,...,...,...
134,0.680294,33.069705,0.742102,0.322021,0.523608,0.014938,22.497115,0.000961,0.000000,37.696154,3.000000
135,0.758193,41.043692,0.722918,0.227705,0.735524,0.011441,31.119023,0.000315,0.000079,52.430907,1.234375
136,0.770113,36.618802,0.652963,0.298398,0.648063,0.020431,28.200620,0.002023,0.000000,42.743421,4.349636
137,0.628897,45.870392,0.724255,0.252893,0.492193,0.019663,28.847736,0.000759,0.000054,51.536395,2.658163


In [136]:
X_new_std

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_MinorAxisLength,glszm_ZonePercentage_PET_b2,LBP_201_PET,shape_Maximum3DDiameter,glszm_GrayLevelVariance_PET_b2
0,-0.485459,0.059912,0.704475,-0.258002,0.064971,1.172200,-0.225260,-0.086297,0.038290,0.030929,0.606009
1,0.666232,-0.755402,0.101791,1.308278,-0.952446,1.947696,-0.603339,1.454382,1.827361,-0.744926,-0.543389
2,0.337179,0.338093,-1.132016,-0.079152,0.444285,-0.783916,0.754279,-0.402730,-0.345359,0.543698,-0.511718
3,0.695653,-0.709822,-0.017443,1.204345,-0.718597,0.165035,-0.522339,-0.218202,-0.345359,-0.564759,-0.678045
4,0.617917,-0.285114,-1.720923,1.789265,-0.496529,-0.323382,0.069107,1.274704,2.133915,-0.266658,-0.571977
...,...,...,...,...,...,...,...,...,...,...,...
134,-0.105628,-0.360532,0.525285,0.206714,-0.004852,-0.151010,-0.455811,-0.163123,-0.345359,-0.358084,-0.074784
135,0.267171,0.011834,0.344965,-0.412334,1.278289,-0.470412,0.257736,-0.733664,0.144386,0.236335,-0.531270
136,0.324219,-0.194798,-0.312609,0.051664,0.748713,0.350720,0.016210,0.773935,-0.345359,-0.154471,0.274151
137,-0.351598,0.237230,0.357529,-0.247011,-0.195069,0.280586,0.069765,-0.341933,-0.008205,0.200249,-0.163163


In [137]:
MAASTRO_new 

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_MinorAxisLength,glszm_ZonePercentage_PET_b2,LBP_201_PET,shape_Maximum3DDiameter,glszm_GrayLevelVariance_PET_b2
0,0.765178,50.002093,0.668072,0.215184,0.610062,0.010495,38.260495,0.001102,0.000026,58.864251,4.473923
1,0.776540,41.753334,0.669961,0.276092,0.504616,0.035018,32.423122,0.000726,0.000167,48.723711,5.207101
2,0.697164,44.375483,0.624081,0.298887,0.478604,0.012819,30.936983,0.001030,0.000000,48.969378,6.728395
3,0.574636,46.115989,0.577624,0.361096,0.446059,0.029974,26.499895,0.000879,0.000080,55.226805,2.694215
4,0.633419,54.394967,0.630933,0.251519,0.480378,0.013458,34.454789,0.000458,0.000035,67.089492,2.710059
...,...,...,...,...,...,...,...,...,...,...,...
94,0.882411,34.218615,0.671754,0.307574,0.577884,0.039092,30.194871,0.003809,0.000078,43.520110,16.942940
95,0.535802,51.046869,0.632189,0.289922,0.455642,0.015831,27.351039,0.001032,0.000054,52.440442,4.742382
96,0.716610,50.417953,0.645548,0.228184,0.631485,0.011031,36.130031,0.000282,0.000028,57.671483,3.250000
97,0.665145,44.901412,0.727488,0.223872,0.628338,0.019801,29.865942,0.000953,0.000000,51.478151,3.222400


In [138]:
MAASTRO_new_std

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_MinorAxisLength,glszm_ZonePercentage_PET_b2,LBP_201_PET,shape_Maximum3DDiameter,glszm_GrayLevelVariance_PET_b2
0,0.300600,0.430171,-0.170593,-0.494510,0.518623,-0.556880,0.848762,-0.039126,-0.182156,0.495864,0.306284
1,0.354974,0.044973,-0.152832,-0.094745,-0.119851,1.683040,0.365663,-0.371303,0.696135,0.086782,0.495840
2,-0.024893,0.167421,-0.584101,0.054869,-0.277351,-0.344618,0.242671,-0.102565,-0.345359,0.096692,0.889156
3,-0.611274,0.248699,-1.020797,0.463178,-0.474407,1.222314,-0.124542,-0.235724,0.151803,0.349125,-0.153842
4,-0.329957,0.635308,-0.519690,-0.256026,-0.266607,-0.286171,0.533803,-0.607066,-0.126027,0.827681,-0.149746
...,...,...,...,...,...,...,...,...,...,...,...
94,0.861639,-0.306881,-0.135979,0.111891,0.323784,2.055216,0.181254,2.349697,0.138162,-0.123138,3.530028
95,-0.797117,0.478960,-0.507883,-0.003972,-0.416383,-0.069508,-0.054101,-0.100944,-0.007528,0.236720,0.375692
96,0.068172,0.449591,-0.382318,-0.409188,0.648334,-0.507885,0.672446,-0.762565,-0.169833,0.447747,-0.010149
97,-0.178126,0.191981,0.387916,-0.437485,0.629283,0.293176,0.154031,-0.170173,-0.345359,0.197899,-0.017285


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [139]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 20:31:03,193] A new study created in memory with name: no-name-d2c634e9-dfb6-457a-a4eb-9378ffcea7b2


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6340425531914894


[I 2024-04-14 20:31:03,774] A new study created in memory with name: no-name-db707001-444b-490a-82e1-543c1269f96f


Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.6652360515021459
[I 2024-04-14 20:31:03,766] Trial 0 finished with value: 0.6586792319760761 and parameters: {}. Best is trial 0 with value: 0.6586792319760761.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6586792319760761], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 3, 240161), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 3, 766396), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6586792319760761


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2745924402154568
Fold 2 IBS: 0.17145676574325394
Fold 3 IBS: 0.21215675308877574
Fold 4 IBS: 0.25357417599446125
Fold 5 IBS: 0.20650162568964595
[I 2024-04-14 20:31:04,587] Trial 0 finished with value: 0.2236563521463187 and parameters: {}. Best is trial 0 with value: 0.2236563521463187.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2236563521463187], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 3, 882913), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 4, 587101), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2236563521463187


In [140]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [141]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.659
train_ibs:  0.224


#### Test

In [142]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [143]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.499
IBS score: 0.288


In [144]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [145]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [146]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 20:31:05,050] A new study created in memory with name: no-name-677e34c2-8c26-4799-8629-e7c915f0a0bb


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7383720930232558
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.6520912547528517


[I 2024-04-14 20:31:06,228] A new study created in memory with name: no-name-59013976-8820-49c5-ad73-85213da67fc9


Fold 5 C-index: 0.6244635193133047
[I 2024-04-14 20:31:06,180] Trial 0 finished with value: 0.6328285538874934 and parameters: {}. Best is trial 0 with value: 0.6328285538874934.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6328285538874934], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 5, 136076), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 6, 179898), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6328285538874934


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709562302793
Fold 2 IBS: 0.2320398715413607
Fold 3 IBS: 0.2289818676351831
Fold 4 IBS: 0.24197476311778662
Fold 5 IBS: 0.22939558654388334
[I 2024-04-14 20:31:07,065] Trial 0 finished with value: 0.23592783689224833 and parameters: {}. Best is trial 0 with value: 0.23592783689224833.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783689224833], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 6, 270305), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 7, 64830), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783689224833


In [147]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [148]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.236


#### Test

In [149]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [150]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.543


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [151]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [152]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 20:31:07,401] A new study created in memory with name: no-name-4a76f6cc-e8c7-4804-9a05-e8ff8223d158


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617


[I 2024-04-14 20:31:08,435] A new study created in memory with name: no-name-f57b0364-9807-46bb-8c22-e2301eedbc1f


Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:08,423] Trial 0 finished with value: 0.6574227098408564 and parameters: {}. Best is trial 0 with value: 0.6574227098408564.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6574227098408564], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 7, 528240), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 8, 423556), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6574227098408564


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.27277647463896704
Fold 2 IBS: 0.17342030361577185
Fold 3 IBS: 0.20977142507972169
Fold 4 IBS: 0.2526073567646622
Fold 5 IBS: 0.2050410788138167
[I 2024-04-14 20:31:09,296] Trial 0 finished with value: 0.22272332778258788 and parameters: {}. Best is trial 0 with value: 0.22272332778258788.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.22272332778258788], datetime_start=datetime.datetime(2024, 4, 14, 20, 31, 8, 516335), datetime_complete=datetime.datetime(2024, 4, 14, 20, 31, 9, 296236), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.22272332778258788


In [153]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [154]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.657
train_ibs:  0.223


#### Test

In [155]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [156]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.498


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [157]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [158]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 20:31:09,812] A new study created in memory with name: no-name-cf6143cd-b8ee-499e-971c-346338c2d039


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6382978723404256
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:10,767] Trial 0 finished with value: 0.6580925585585977 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6580925585585977.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:11,574] Trial 1 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6581831661146207.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:12,203] Trial 2 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:26,548] Trial 24 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.8896057121430163}. Best is trial 4 with value: 0.6589436223883849.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:27,040] Trial 25 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.6172146917600154}. Best is trial 4 with value: 0.6589436223883849.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:27,550] Trial 26 finished with value: 0.6589436223883849 and parameters: {'l1_ratio': 0.7259873255784245}. Best is trial 4 with value: 0.658943

Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:37,879] Trial 48 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.6273814473795578}. Best is trial 4 with value: 0.6589436223883849.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6382978723404256
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:38,345] Trial 49 finished with value: 0.6565716460110693 and parameters: {'l1_ratio': 0.9431560437165498}. Best is trial 4 with value: 0.6589436223883849.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 20:31:38,825] Trial 50 finished with value: 0.6581831661146207 and parameters: {'l1_ratio': 0.8805577934156339}. Best is trial 4 with value: 0.658943

...

Fold 1 IBS: 0.27267447509122855
Fold 2 IBS: 0.17344925973192413
Fold 3 IBS: 0.20995771934819113
Fold 4 IBS: 0.2523770813863882
Fold 5 IBS: 0.20507943055313463
[I 2024-04-14 20:32:07,985] Trial 0 finished with value: 0.22270759322217332 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.22270759322217332.
Fold 1 IBS: 0.2725025347145614
Fold 2 IBS: 0.17314954653948833
Fold 3 IBS: 0.2100613142241499
Fold 4 IBS: 0.25224126142504594
Fold 5 IBS: 0.20489681131134482
[I 2024-04-14 20:32:08,750] Trial 1 finished with value: 0.22257029364291805 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.22257029364291805.
Fold 1 IBS: 0.27243658214766786
Fold 2 IBS: 0.17282261994180062
Fold 3 IBS: 0.20988632452153375
Fold 4 IBS: 0.2521745411667705
Fold 5 IBS: 0.20478516930358523
[I 2024-04-14 20:32:09,496] Trial 2 finished with value: 0.2224210474162716 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.22242104741627

Fold 1 IBS: 0.27253376544295665
Fold 2 IBS: 0.17240489453662233
Fold 3 IBS: 0.21002324567043654
Fold 4 IBS: 0.2520836800787669
Fold 5 IBS: 0.20483452937549124
[I 2024-04-14 20:32:24,704] Trial 25 finished with value: 0.22237602302085474 and parameters: {'l1_ratio': 0.1238911155354028}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.27280705714960496
Fold 2 IBS: 0.1734416623532437
Fold 3 IBS: 0.20982839720989
Fold 4 IBS: 0.25260209853835824
Fold 5 IBS: 0.20534754660191004
[I 2024-04-14 20:32:26,119] Trial 26 finished with value: 0.22280535237060137 and parameters: {'l1_ratio': 0.9276289503411492}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.2723972152384981
Fold 2 IBS: 0.17233541800588956
Fold 3 IBS: 0.22874926185250916
Fold 4 IBS: 0.2520723565594682
Fold 5 IBS: 0.20485489827924006
[I 2024-04-14 20:32:26,702] Trial 27 finished with value: 0.226081829987121 and parameters: {'l1_ratio': 0.08555734699384382}. Best is trial 21 with value: 0.2223256926731

Fold 1 IBS: 0.2725595777761324
Fold 2 IBS: 0.22773614218874613
Fold 3 IBS: 0.22884747321581533
Fold 4 IBS: 0.23930457631194157
Fold 5 IBS: 0.22719631294170334
[I 2024-04-14 20:32:42,136] Trial 50 finished with value: 0.23912881648686776 and parameters: {'l1_ratio': 0.03738779124899358}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.27241324639370174
Fold 2 IBS: 0.17240363993199545
Fold 3 IBS: 0.21004401066910072
Fold 4 IBS: 0.2521006210760798
Fold 5 IBS: 0.20485143916669354
[I 2024-04-14 20:32:43,045] Trial 51 finished with value: 0.22236259144751425 and parameters: {'l1_ratio': 0.12594937537013567}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.2725807308561395
Fold 2 IBS: 0.1725358755395789
Fold 3 IBS: 0.21006056510271615
Fold 4 IBS: 0.2521299293563059
Fold 5 IBS: 0.20479134153756906
[I 2024-04-14 20:32:43,846] Trial 52 finished with value: 0.22241968847846189 and parameters: {'l1_ratio': 0.1547276572252059}. Best is trial 21 with value: 0.22232569

Fold 5 IBS: 0.20479804430809104
[I 2024-04-14 20:32:56,735] Trial 74 finished with value: 0.22243043519060263 and parameters: {'l1_ratio': 0.17063221060123465}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.27252024947484704
Fold 2 IBS: 0.1732585874278285
Fold 3 IBS: 0.20975567436238726
Fold 4 IBS: 0.25226631721901643
Fold 5 IBS: 0.204898114726913
[I 2024-04-14 20:32:57,274] Trial 75 finished with value: 0.22253978864219848 and parameters: {'l1_ratio': 0.3186692068904706}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.2724317016287503
Fold 2 IBS: 0.17281995720978222
Fold 3 IBS: 0.20988153305424986
Fold 4 IBS: 0.252171936742705
Fold 5 IBS: 0.20479798026846704
[I 2024-04-14 20:32:57,904] Trial 76 finished with value: 0.22242062178079086 and parameters: {'l1_ratio': 0.22627165809269023}. Best is trial 21 with value: 0.22232569267316699.
Fold 1 IBS: 0.2724137644671228
Fold 2 IBS: 0.17307016239184442
Fold 3 IBS: 0.20982935102349748
Fold 4 IBS: 0.252182275

Fold 1 IBS: 0.27251450716401543
Fold 2 IBS: 0.17269346713002628
Fold 3 IBS: 0.20982783116730855
Fold 4 IBS: 0.25211901168401063
Fold 5 IBS: 0.20482923375301645
[I 2024-04-14 20:33:14,530] Trial 99 finished with value: 0.22239681017967547 and parameters: {'l1_ratio': 0.19742502235371856}. Best is trial 21 with value: 0.22232569267316699.


* Best trial for IBS: 
 FrozenTrial(number=21, state=TrialState.COMPLETE, values=[0.22232569267316699], datetime_start=datetime.datetime(2024, 4, 14, 20, 32, 21, 405235), datetime_complete=datetime.datetime(2024, 4, 14, 20, 32, 22, 200613), params={'l1_ratio': 0.11781148032178282}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=21, value=None)


* Best Score for IBS: 
 0.22232569267316699


In [159]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [160]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.659
train_ibs:  0.222


#### Test

In [161]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [162]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.7194970228885845)

test_cindex : 0.497


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.11781148032178282)

test_ibs:  0.288


In [163]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [164]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 20:33:15,028] A new study created in memory with name: no-name-aeb9bb76-54d9-440f-b17a-8ed627906103


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6892430278884463
Fold 2 C-index: 0.7790697674418605
Fold 3 C-index: 0.6340425531914894
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 20:33:21,432] Trial 0 finished with value: 0.7036219044111919 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7036219044111919.
Fold 1 C-index: 0.6533864541832669
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.752851711026616
Fold 5 C-index: 0.7081545064377682
[I 2024-04-14 20:33:25,324] Trial 1 finished with value: 0.7031589235757779 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 5 C-index: 0.6587982832618026
[I 2024-04-14 20:34:10,306] Trial 15 finished with value: 0.6741633120242068 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 2, 'min_samples_leaf': 7, 'max_depth': 14, 'n_estimators': 79, 'oob_score': False, 'max_samples': 0.9887043893136873, 'max_features': None, 'min_weight_fraction_leaf': 0.11311294291919405, 'warm_start': False}. Best is trial 2 with value: 0.7068386430750734.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.7300380228136882
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 20:34:16,945] Trial 16 finished with value: 0.6950964351319479 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 14, 'max_depth': 14, 'n_estimators': 475, 'oob_score': False, 'max_samples': 0.6976003666324501, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.22594306843997863, 'warm_start': False}. Best is trial 2 with value: 0.706838643075

Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.7659574468085106
Fold 4 C-index: 0.7756653992395437
Fold 5 C-index: 0.7896995708154506
[I 2024-04-14 20:35:05,620] Trial 30 finished with value: 0.7495900956522649 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 1, 'n_estimators': 158, 'oob_score': True, 'max_samples': 0.6787918855242834, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.010889063772751496, 'warm_start': True}. Best is trial 29 with value: 0.7732179335818625.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.7939914163090128
[I 2024-04-14 20:35:07,586] Trial 31 finished with value: 0.7702273294735654 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 11, 'min_samples_leaf': 12, 'max_depth': 3, 'n_estimators': 154, 'oob_score': True, 'max_samples': 0.601481145304731

...

Fold 5 IBS: 0.20628342384593995
[I 2024-04-14 20:37:42,048] Trial 31 finished with value: 0.20677807408632284 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 8, 'n_estimators': 241, 'oob_score': False, 'max_samples': 0.9378460826208653, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1726892235544012}. Best is trial 13 with value: 0.20611217573172563.
Fold 1 IBS: 0.22725825690793014
Fold 2 IBS: 0.18682969848680658
Fold 3 IBS: 0.21075711994523577
Fold 4 IBS: 0.20108520766951074
Fold 5 IBS: 0.20700698808236373
[I 2024-04-14 20:37:45,210] Trial 32 finished with value: 0.20658745421836944 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 253, 'oob_score': False, 'max_samples': 0.9502203520505428, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.14645952871104895}. Best is trial 13 with value: 0.20611217573172563.
Fold 1 IBS: 0.2255625199718266
Fold 2 IBS: 0.18

Fold 1 IBS: 0.21901722906427928
Fold 2 IBS: 0.1880121546737221
Fold 3 IBS: 0.2143915775970178
Fold 4 IBS: 0.20859975309235104
Fold 5 IBS: 0.21293055714902465
[I 2024-04-14 20:38:41,551] Trial 47 finished with value: 0.20859025431527894 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 17, 'max_depth': 10, 'n_estimators': 121, 'oob_score': True, 'max_samples': 0.7066230764403721, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05497537161420828}. Best is trial 13 with value: 0.20611217573172563.
Fold 1 IBS: 0.24646474217837117
Fold 2 IBS: 0.2321704166482232
Fold 3 IBS: 0.22960401808871153
Fold 4 IBS: 0.24144503779242357
Fold 5 IBS: 0.23032179092673608
[I 2024-04-14 20:38:45,418] Trial 48 finished with value: 0.2360012011268931 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 4, 'n_estimators': 334, 'oob_score': False, 'max_samples': 0.8517947741571208, 'max_features': 'auto', 'min_weight_fraction_le

Fold 5 IBS: 0.20613396395481967
[I 2024-04-14 20:39:30,244] Trial 62 finished with value: 0.20649181863511998 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 241, 'oob_score': False, 'max_samples': 0.9254593391142755, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18664631824303496}. Best is trial 13 with value: 0.20611217573172563.
Fold 1 IBS: 0.21846829565911444
Fold 2 IBS: 0.18601729779341092
Fold 3 IBS: 0.21422746242483537
Fold 4 IBS: 0.20884474697455188
Fold 5 IBS: 0.20749656516108242
[I 2024-04-14 20:39:33,422] Trial 63 finished with value: 0.207010873602599 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 10, 'min_samples_leaf': 17, 'max_depth': 7, 'n_estimators': 246, 'oob_score': False, 'max_samples': 0.8562041421528351, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.15233334363188203}. Best is trial 13 with value: 0.20611217573172563.
Fold 1 IBS: 0.22160509975134576
Fold 2 IBS: 0.

Fold 1 IBS: 0.22522734682714227
Fold 2 IBS: 0.19837387736388737
Fold 3 IBS: 0.21858863686312044
Fold 4 IBS: 0.22243847472975584
Fold 5 IBS: 0.21919229133701493
[I 2024-04-14 20:40:14,966] Trial 78 finished with value: 0.2167641254241842 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 11, 'max_depth': 14, 'n_estimators': 51, 'oob_score': False, 'max_samples': 0.8217049327242889, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.28987424398930645}. Best is trial 72 with value: 0.2056332376262735.
Fold 1 IBS: 0.24583066360819886
Fold 2 IBS: 0.2321884833969819
Fold 3 IBS: 0.22989598582612408
Fold 4 IBS: 0.2412239000663218
Fold 5 IBS: 0.2298832890136102
[I 2024-04-14 20:40:16,982] Trial 79 finished with value: 0.23580446438224736 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 9, 'max_depth': 10, 'n_estimators': 150, 'oob_score': False, 'max_samples': 0.12074719353470553, 'max_features': 'auto', 'min_weight_fraction_le

Fold 1 IBS: 0.21916825027396117
Fold 2 IBS: 0.18609727919340027
Fold 3 IBS: 0.2128086280899691
Fold 4 IBS: 0.20635539721723173
Fold 5 IBS: 0.20776868063059972
[I 2024-04-14 20:40:59,245] Trial 94 finished with value: 0.2064396470810324 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 254, 'oob_score': False, 'max_samples': 0.8841008918901287, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.21456685175612356}. Best is trial 72 with value: 0.2056332376262735.
Fold 1 IBS: 0.23182805456625516
Fold 2 IBS: 0.2022942845600236
Fold 3 IBS: 0.222033047775834
Fold 4 IBS: 0.22231079464762368
Fold 5 IBS: 0.2184937117905702
[I 2024-04-14 20:41:02,855] Trial 95 finished with value: 0.21939197866806132 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 17, 'max_depth': 6, 'n_estimators': 279, 'oob_score': False, 'max_samples': 0.5739374614153154, 'max_features': 'log2', 'min_weight_fraction_le

In [165]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [166]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.795
train_ibs:  0.206


#### Test

In [167]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [168]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features='auto', max_leaf_nodes=17,
                     max_samples=0.9538061740605386, min_samples_leaf=11,
                     min_samples_split=4,
                     min_weight_fraction_leaf=0.09899137909936344,
                     n_estimators=40, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.509


RandomSurvivalForest(max_depth=9, max_features='log2', max_leaf_nodes=12,
                     max_samples=0.8661182997981863, min_samples_leaf=10,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.18591741258051608,
                     n_estimators=90, random_state=123)

test_ibs:  0.25


In [169]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [170]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [171]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 20:41:20,410] A new study created in memory with name: no-name-579551a3-995b-4378-82bd-2eb6d66bb7b2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.7957446808510639
Fold 4 C-index: 0.7832699619771863
Fold 5 C-index: 0.7982832618025751
[I 2024-04-14 20:41:22,366] Trial 0 finished with value: 0.7444641826742117 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7444641826742117.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:41:26,094] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.7262357414448669
Fold 5 C-index: 0.7467811158798283
[I 2024-04-14 20:42:01,179] Trial 15 finished with value: 0.7240470547558558 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7621811621627721.
Fold 1 C-index: 0.5099601593625498
Fold 2 C-index: 0.7790697674418605
Fold 3 C-index: 0.7340425531914894
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6351931330472103
[I 2024-04-14 20:42:03,133] Trial 16 finished with value: 0.6662538830648959 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_feat

Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.8212927756653993
Fold 5 C-index: 0.8025751072961373
[I 2024-04-14 20:42:20,159] Trial 30 finished with value: 0.7622070063238183 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 111, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.733146369663002, 'min_weight_fraction_leaf': 0.10323731749624541}. Best is trial 23 with value: 0.7795288364216049.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7790697674418605
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.8365019011406845
Fold 5 C-index: 0.8068669527896996
[I 2024-04-14 20:42:21,113] Trial 31 finished with value: 0.7762873343448058 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 276, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.24158942477782158
Fold 2 IBS: 0.21929391721716995
Fold 3 IBS: 0.2193550775567237
Fold 4 IBS: 0.2323195982287675
Fold 5 IBS: 0.2197126913890985
[I 2024-04-14 20:46:06,402] Trial 30 finished with value: 0.22645414183391624 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.21373780291880431.
Fold 1 IBS: 0.2354981935315128
Fold 2 IBS: 0.20611350006679782
Fold 3 IBS: 0.2080374995849842
Fold 4 IBS: 0.22144619637606308
Fold 5 IBS: 0.21166974370445468
[I 2024-04-14 20:46:11,508] Trial 31 finished with value: 0.21655302665276252 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7

Fold 1 IBS: 0.23004762363280065
Fold 2 IBS: 0.20207981520408078
Fold 3 IBS: 0.20542350510528304
Fold 4 IBS: 0.21487462055621293
Fold 5 IBS: 0.2031286821836111
[I 2024-04-14 20:46:57,700] Trial 45 finished with value: 0.2111108493363977 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 114, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.5545433281929569, 'min_weight_fraction_leaf': 0.022411260801931558}. Best is trial 42 with value: 0.21028638136422434.
Fold 1 IBS: 0.23041156618088032
Fold 2 IBS: 0.20438661273089367
Fold 3 IBS: 0.2083274286291039
Fold 4 IBS: 0.21434598556991874
Fold 5 IBS: 0.20018044432936546
[I 2024-04-14 20:46:59,130] Trial 46 finished with value: 0.21153040748803242 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 102, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.22886388120004955
Fold 2 IBS: 0.1922779447217556
Fold 3 IBS: 0.20596346109163932
Fold 4 IBS: 0.2162435591402912
Fold 5 IBS: 0.20842350045875987
[I 2024-04-14 20:47:28,127] Trial 60 finished with value: 0.2103544693224991 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 194, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.6240740459164607, 'min_weight_fraction_leaf': 0.0999675321939971}. Best is trial 56 with value: 0.20836801003350747.
Fold 1 IBS: 0.2347250098325648
Fold 2 IBS: 0.1889247237462346
Fold 3 IBS: 0.20786146399799835
Fold 4 IBS: 0.21442044305481517
Fold 5 IBS: 0.2103485414591851
[I 2024-04-14 20:47:30,837] Trial 61 finished with value: 0.21125603641815957 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 189, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.664435077

Fold 1 IBS: 0.2323383037155898
Fold 2 IBS: 0.1926386768769483
Fold 3 IBS: 0.2043263234125086
Fold 4 IBS: 0.2177958054014891
Fold 5 IBS: 0.20847206331237897
[I 2024-04-14 20:48:12,847] Trial 75 finished with value: 0.21111423454378295 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 2, 'n_estimators': 248, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.7455530824826245, 'min_weight_fraction_leaf': 0.10174967743425128}. Best is trial 72 with value: 0.2080376531858185.
Fold 1 IBS: 0.23125321916750102
Fold 2 IBS: 0.19448300187171705
Fold 3 IBS: 0.2064500233945674
Fold 4 IBS: 0.2118211264767168
Fold 5 IBS: 0.2062701779464504
[I 2024-04-14 20:48:17,024] Trial 76 finished with value: 0.21005550977139054 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 4, 'n_estimators': 303, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.7855036521

Fold 1 IBS: 0.23068196697328497
Fold 2 IBS: 0.19089408813814912
Fold 3 IBS: 0.20382905085626038
Fold 4 IBS: 0.21084183448318802
Fold 5 IBS: 0.2025699645464283
[I 2024-04-14 20:49:10,754] Trial 90 finished with value: 0.20776338099946218 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 9, 'min_samples_leaf': 5, 'max_depth': 8, 'n_estimators': 269, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.5784776864026564, 'min_weight_fraction_leaf': 0.061135549090751994}. Best is trial 90 with value: 0.20776338099946218.
Fold 1 IBS: 0.2310128688684838
Fold 2 IBS: 0.19092846353894036
Fold 3 IBS: 0.20422692176342377
Fold 4 IBS: 0.21039728779594005
Fold 5 IBS: 0.2025066026589034
[I 2024-04-14 20:49:17,187] Trial 91 finished with value: 0.2078144289251383 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 9, 'min_samples_leaf': 5, 'max_depth': 8, 'n_estimators': 273, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.5774200

In [172]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [173]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.794
train_ibs:  0.207


#### Test

In [174]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [175]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=8, max_features=None, max_leaf_nodes=14,
                   max_samples=0.8680829438458175, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.05559928273470952,
                   n_estimators=87, random_state=123, warm_start=True)

C-index score: 0.506


ExtraSurvivalTrees(max_depth=9, max_features=None, max_leaf_nodes=9,
                   max_samples=0.598575294078392, min_samples_leaf=5,
                   min_samples_split=3,
                   min_weight_fraction_leaf=0.056138512874115615,
                   n_estimators=334, oob_score=True, random_state=123)

IBS: 0.257


In [176]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [177]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 20:50:01,966] A new study created in memory with name: no-name-bc61b3c3-4c97-429f-a89d-52cc7c89b3b4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:50:29,984] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:50:47,598] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:58:22,280] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6985461379550671.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:59:26,381] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 21:10:46,856] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6985461379550671.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 21:11:17,554] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:05:16,875] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.20419816136861105, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 1.5125790196698774, 'min_weight_fraction_leaf': 0.21953014805517473, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.9659902853562541, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 2}. Best is trial 41 with value: 0.23444244767806421.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:05:37,149] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2887765610954624, 'learning_rate': 0.09850922

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809254
[I 2024-04-14 23:11:07,200] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.41194318384993134, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.22834912563776077, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893850403970479, 'min_weight_fraction_leaf': 0.11486333706731977, 'max_features': 'auto', 'min_impurity_decrease': 5.977994289888799e-05, 'validation_fraction': 0.5200975528135579, 'min_samples_split': 18, 'max_leaf_nodes': 8, 'min_samples_leaf': 6, 'max_depth': 4}. Best is trial 41 with value: 0.23444244767806421.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:11:43,316] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6282300518309584, 'learning_rate': 0.003920668

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:16:29,209] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.662142493553532, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.21207366660855861, 'n_estimators': 446, 'criterion': 'squared_error', 'ccp_alpha': 1.2832467794713243, 'min_weight_fraction_leaf': 0.13484676844178797, 'max_features': None, 'min_impurity_decrease': 5.433160474458954e-07, 'validation_fraction': 0.3082779826259943, 'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 72 with value: 0.23383844378939234.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939128724118643
[I 2024-04-14 23:17:02,943] Trial 78 finished with value: 0.2359269823509495 and parameters: {'subsample': 0.6029614475469536, 'learning_rate': 0.02742354559286

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:22:39,457] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6723499933201714, 'learning_rate': 0.08445478533817788, 'dropout_rate': 0.2193630306533548, 'n_estimators': 489, 'criterion': 'squared_error', 'ccp_alpha': 0.5761857609766958, 'min_weight_fraction_leaf': 0.09845759507035474, 'max_features': None, 'min_impurity_decrease': 0.0001627481847856609, 'validation_fraction': 0.16955200144561516, 'min_samples_split': 5, 'max_leaf_nodes': 20, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 81 with value: 0.22584174306952026.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:23:13,252] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7153666202558423, 'learning_rate': 0.0596510546088

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:27:45,550] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.679529118780463, 'learning_rate': 0.09119413615927978, 'dropout_rate': 0.10216550214025771, 'n_estimators': 411, 'criterion': 'friedman_mse', 'ccp_alpha': 1.716394824031096, 'min_weight_fraction_leaf': 0.07701366240523831, 'max_features': None, 'min_impurity_decrease': 9.975932931017948e-07, 'validation_fraction': 0.46082617508021007, 'min_samples_split': 11, 'max_leaf_nodes': 19, 'min_samples_leaf': 14, 'max_depth': 3}. Best is trial 98 with value: 0.22262159783338803.


* Best trial for IBS: 
 FrozenTrial(number=98, state=TrialState.COMPLETE, values=[0.22262159783338803], datetime_start=datetime.datetime(2024, 4, 14, 23, 26, 40, 407865), datetime_complete=datetime.datetime(2024, 4, 14, 23, 27, 14, 778330), params={'subsample': 0.6871257727531467, 'learning_rate': 0.0903541265036734

In [178]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [179]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.738
train_ibs:  0.223


#### Test

In [180]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [181]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02313127962403605,
                                 criterion='squared_error',
                                 dropout_rate=0.3476599176890519,
                                 learning_rate=0.03772869943983537, max_depth=4,
                                 max_features=1, max_leaf_nodes=18,
                                 min_impurity_decrease=1.0180375358163635e-06,
                                 min_samples_leaf=18, min_samples_split=18,
                                 min_weight_fraction_leaf=0.2716607703914049,
                                 n_estimators=392, random_state=123,
                                 subsample=0.9115482308909325,
                                 validation_fraction=0.9423719780700139)

C-index score: 0.514


GradientBoostingSurvivalAnalysis(ccp_alpha=0.050290828631146894,
                                 dropout_rate=0.1154715714520811,
                                 learning_rate=0.09035412650367348,
                                 max_leaf_nodes=18,
                                 min_impurity_decrease=8.847299946943358e-07,
                                 min_samples_leaf=13, min_samples_split=11,
                                 min_weight_fraction_leaf=0.0717973518747433,
                                 n_estimators=413, random_state=123,
                                 subsample=0.6871257727531467,
                                 validation_fraction=0.4879609273935703)

IBS: 0.231


In [182]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [183]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [184]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 23:27:58,507] A new study created in memory with name: no-name-0044fd14-e42f-4c16-89d8-efd423c8015e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 23:27:59,201] Trial 0 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 23:28:03,166] Trial 1 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 23:28:38,674] Trial 19 finished with value: 0.6443949846669678 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.6482706126490928.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 23:28:42,144] Trial 20 finished with value: 0.6399802791763081 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.6482706126490928.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.6738197424892703


Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 23:29:30,037] Trial 38 finished with value: 0.631313522170144 and parameters: {'subsample': 0.3297325755614151, 'dropout_rate': 0.8867789132968811, 'n_estimators': 392, 'learning_rate': 0.08480908032505553}. Best is trial 11 with value: 0.6482706126490928.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 23:29:30,688] Trial 39 finished with value: 0.6383467162791461 and parameters: {'subsample': 0.1910828860959901, 'dropout_rate': 0.29848665859072016, 'n_estimators': 145, 'learning_rate': 0.06892741183938003}. Best is trial 11 with value: 0.6482706126490928.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
F

...

Fold 4 IBS: 0.2403770465027076
Fold 5 IBS: 0.22389919673997055
[I 2024-04-14 23:32:38,911] Trial 19 finished with value: 0.24208951118640676 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24532728078212304
Fold 2 IBS: 0.21338053568843193
Fold 3 IBS: 0.23280583204014674
Fold 4 IBS: 0.23050963160208696
Fold 5 IBS: 0.21229213294998353
[I 2024-04-14 23:32:39,243] Trial 20 finished with value: 0.22686308261255445 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24480948267779246
Fold 2 IBS: 0.217894217650721
Fold 3 IBS: 0.23043919128917217
Fold 4 IBS: 0.23275995527788665
Fold 5 IBS: 0.215372968667489
[I 2024-04-14 23:32:39,527] Trial 21 finished with value: 0.22825516311261

Fold 5 IBS: 0.21475575962300342
[I 2024-04-14 23:32:51,033] Trial 38 finished with value: 0.23328897274995045 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.29108106719697524
Fold 2 IBS: 0.21415810554809628
Fold 3 IBS: 0.2917820507930866
Fold 4 IBS: 0.2588982634047282
Fold 5 IBS: 0.24407574900323586
[I 2024-04-14 23:32:51,390] Trial 39 finished with value: 0.25999904718922445 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.2925659515921918
Fold 2 IBS: 0.2140746731129848
Fold 3 IBS: 0.2919432513336724
Fold 4 IBS: 0.257477121073266
Fold 5 IBS: 0.24657769529804316
[I 2024-04-14 23:32:51,903] Trial 40 finished with value: 0.2605277384820316 and parameters: {'subsample': 0.

Fold 4 IBS: 0.2292932477615486
Fold 5 IBS: 0.2122783863810015
[I 2024-04-14 23:33:01,368] Trial 58 finished with value: 0.2284139145750997 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2450009034050451
Fold 2 IBS: 0.21574021216390366
Fold 3 IBS: 0.2316126494096215
Fold 4 IBS: 0.2313085066143514
Fold 5 IBS: 0.2135215585555951
[I 2024-04-14 23:33:01,553] Trial 59 finished with value: 0.22743676602970336 and parameters: {'subsample': 0.7169412850975417, 'dropout_rate': 0.7863461136932807, 'n_estimators': 20, 'learning_rate': 0.03702429542217137}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.27643234236120234
Fold 2 IBS: 0.20531960631582807
Fold 3 IBS: 0.2756328535259464
Fold 4 IBS: 0.24566178547120296
Fold 5 IBS: 0.23379390635926353
[I 2024-04-14 23:33:02,108] Trial 60 finished with value: 0.24736809880668864 

Fold 4 IBS: 0.2283240329592914
Fold 5 IBS: 0.21054396005104126
[I 2024-04-14 23:33:16,632] Trial 77 finished with value: 0.22739933013100408 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24684927097070503
Fold 2 IBS: 0.2094952280230178
Fold 3 IBS: 0.23652719105524916
Fold 4 IBS: 0.22913951054643386
Fold 5 IBS: 0.2106573202990221
[I 2024-04-14 23:33:16,949] Trial 78 finished with value: 0.2265337041788856 and parameters: {'subsample': 0.5939500427136554, 'dropout_rate': 0.7406477151463077, 'n_estimators': 73, 'learning_rate': 0.015420072941943445}. Best is trial 78 with value: 0.2265337041788856.
Fold 1 IBS: 0.24502409439696052
Fold 2 IBS: 0.22205787317352343
Fold 3 IBS: 0.22926533447627775
Fold 4 IBS: 0.2352795597651623
Fold 5 IBS: 0.2188730593846164
[I 2024-04-14 23:33:17,282] Trial 79 finished with value: 0.2300999842393

Fold 4 IBS: 0.22741639083301954
Fold 5 IBS: 0.20957521984090532
[I 2024-04-14 23:33:26,189] Trial 96 finished with value: 0.2262940262454026 and parameters: {'subsample': 0.34523349906195283, 'dropout_rate': 0.7739280807288145, 'n_estimators': 132, 'learning_rate': 0.010648288131411126}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24385115968166024
Fold 2 IBS: 0.21643646571599923
Fold 3 IBS: 0.23116557489710682
Fold 4 IBS: 0.23167317839356094
Fold 5 IBS: 0.21516282794158326
[I 2024-04-14 23:33:27,248] Trial 97 finished with value: 0.22765784132598207 and parameters: {'subsample': 0.37469859653908655, 'dropout_rate': 0.8528934112321543, 'n_estimators': 187, 'learning_rate': 0.003684889317176295}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24621689559308768
Fold 2 IBS: 0.2055572040394727
Fold 3 IBS: 0.24230333094691225
Fold 4 IBS: 0.22648768747110257
Fold 5 IBS: 0.20952956681329585
[I 2024-04-14 23:33:28,561] Trial 98 finished with value: 0.226018

In [185]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [186]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.65
train_ibs:  0.224


#### Test

In [187]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [188]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.2525866290623745,
                                              learning_rate=0.0801524070565027,
                                              n_estimators=446,
                                              random_state=123,
                                              subsample=0.10110319978064844)

C-index score: 0.532


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7782160113329367,
                                              learning_rate=0.005308151321564225,
                                              n_estimators=212,
                                              random_state=123,
                                              subsample=0.17778434918103095)

IBS: 0.233


In [189]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [190]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.795,1.0
ExtraSurvivalTrees,0.794,2.0
GradientBoosting,0.738,3.0
CoxPH,0.659,4.5
CoxElastic,0.659,4.5
CoxLasso,0.657,6.0
ComponentwiseGradientBoosting,0.650,7.0
CoxRidge,0.633,8.0


In [191]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.206,1.0
ExtraSurvivalTrees,0.207,2.0
CoxElastic,0.222,3.0
CoxLasso,0.223,4.5
GradientBoosting,0.223,4.5
CoxPH,0.224,6.5
ComponentwiseGradientBoosting,0.224,6.5
CoxRidge,0.236,8.0


In [192]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.543,1.0
ComponentwiseGradientBoosting,0.532,2.0
GradientBoosting,0.514,3.0
Randomsurvivalforest,0.509,4.0
ExtraSurvivalTrees,0.506,5.0
CoxPH,0.499,6.0
CoxLasso,0.498,7.0
CoxElastic,0.497,8.0


In [193]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.0
GradientBoosting,0.231,2.0
ComponentwiseGradientBoosting,0.233,3.0
Randomsurvivalforest,0.250,4.0
ExtraSurvivalTrees,0.257,5.0
CoxPH,0.288,7.0
CoxLasso,0.288,7.0
CoxElastic,0.288,7.0


In [194]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/robust/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_robust_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [195]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
